# 02 — Partie 1 : Modèles de Séquences en PyTorch (`RNN`, `Bi-RNN`, `GRU`, `LSTM`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassinekamouss/fstt-dl-lab3-nlp-pytorch/blob/main/notebooks/02_part1_sequence_models.ipynb)

**Université Abdelmalek Essaadi — FST Tanger — Master MBD — Deep Learning (Lab 3)**

Ce notebook implémente, optimise et évalue les quatre architectures séquentielles demandées :
1. **`RNN`** (unidirectionnel)
2. **`Bidirectional RNN`** (`Bi-RNN`)
3. **`GRU`**
4. **`LSTM`**

Étapes couvertes :
- Chargement des splits anti-leakage et du vocabulaire générés par `01_scraping_and_dataset.ipynb`.
- Vérification des architectures (`pack_padded_sequence`, pooling masqué, `LayerNorm`).
- Recherche d'hyperparamètres avec **Optuna** (`tuning_results.csv`).
- Entraînement final (`AdamW`, class weights, gradient clipping, `ReduceLROnPlateau`, `EarlyStopping`).
- Évaluation complète sur `test` : **Accuracy, Precision, Recall, Macro-F1, Weighted-F1, ROC-AUC, Matrices de confusion, Analyse d'erreurs**.
- **Benchmark compagnon Language Modeling & Score BLEU** (Perplexity + BLEU-1/2/4 / SacreBLEU).

## 1. Bootstrap Google Colab / Drive & Chargement des Données

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None
REPO_URL = "https://github.com/yassinekamouss/fstt-dl-lab3-nlp-pytorch.git"
REPO_NAME = "fstt-dl-lab3-nlp-pytorch"
USE_DRIVE = True

if IN_COLAB:
    repo_dir = Path("/content") / REPO_NAME
    if not repo_dir.exists():
        !git clone {REPO_URL} {repo_dir}
    else:
        !git -C {repo_dir} pull --ff-only
    os.chdir(repo_dir)
else:
    cwd = Path.cwd().resolve()
    repo_dir = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

In [ ]:
if IN_COLAB:
    !pip install -q optuna sacrebleu arabic-reshaper python-bidi pyyaml PyArabic nltk

import json
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.colab_utils import get_device, load_config, print_system_info, seed_everything, setup_environment
from src.dataset import ArabicClassificationDataset, PAD_IDX, Vocabulary, classification_collate_fn

paths = setup_environment(
    use_drive=USE_DRIVE,
    drive_dir="/content/drive/MyDrive/fstt-dl-lab3-nlp-pytorch",
    repo_root=Path.cwd(),
)
cfg = load_config(paths["config"])
seed_everything(cfg["project"]["seed"])
device = get_device()
print_system_info()

train_df = pd.read_csv(paths["data_processed"] / "train.csv")
val_df = pd.read_csv(paths["data_processed"] / "val.csv")
test_df = pd.read_csv(paths["data_processed"] / "test.csv")
vocab = Vocabulary.load(paths["data_processed"] / "vocab.json")

text_col = cfg["preprocessing"]["representation_for_models"]
class_names = cfg["preprocessing"]["class_names"]
num_classes = len(class_names)
max_seq_len = int(cfg["preprocessing"]["max_seq_len"])

print(f"Splits charges -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Vocab: {len(vocab)}")

## 2. Vérification des 4 Architectures PyTorch (`SequenceClassifier`)

Toutes les architectures exploitent `nn.Embedding(padding_idx=0)`, `pack_padded_sequence` sur les longueurs réelles des séquences, un pooling masqué (`last`, `mean`, ou `mean_max`), `LayerNorm` et une couche linéaire finale.

In [ ]:
from src.models import build_classifier

sample_ds = ArabicClassificationDataset(
    train_df[text_col].astype(str).tolist()[:8],
    train_df["label"].astype(int).tolist()[:8],
    vocab=vocab,
    max_len=max_seq_len,
)
sample_loader = DataLoader(sample_ds, batch_size=4, collate_fn=classification_collate_fn)
x_batch, lengths_batch, y_batch = next(iter(sample_loader))

for arch in cfg["part1_models"]["architectures"]:
    m = build_classifier(arch=arch, vocab_size=len(vocab), num_classes=num_classes, pad_idx=PAD_IDX)
    out = m(x_batch, lengths_batch)
    print(f"{arch.upper():<6} | params={m.count_parameters():>9,} | output_shape={tuple(out.shape)}")

## 3. Tuning des Hyperparamètres avec Optuna

Pour chaque architecture (`rnn`, `birnn`, `gru`, `lstm`), nous optimisons le **Macro-F1 validation** sur l'espace : `embedding_dim`, `hidden_dim`, `num_layers`, `dropout`, `lr`, `batch_size`, `pooling`.

In [ ]:
from src.trainer import tune_architecture

best_configs = {}
all_trials = []

for arch in cfg["part1_models"]["architectures"]:
    print(f"\n>>> Tuning Optuna pour {arch.upper()}...")
    best_params, trials_df = tune_architecture(
        arch=arch,
        train_df=train_df,
        val_df=val_df,
        vocab=vocab,
        cfg=cfg,
        device=device,
        text_col=text_col,
    )
    best_configs[arch] = best_params
    all_trials.append(trials_df)
    print(f"Meilleurs hyperparametres {arch.upper()} (val_f1={trials_df.iloc[0]['best_val_f1']:.4f}): {best_params}")

tuning_df = pd.concat(all_trials, ignore_index=True)
tuning_df.to_csv(paths["reports"] / "tuning_results.csv", index=False)
(paths["reports"] / "best_hyperparams.json").write_text(
    json.dumps(best_configs, indent=2), encoding="utf-8"
)
display(tuning_df.groupby("arch").head(2))

## 4. Entraînement Final des 4 Modèles Optimisés

Chaque modèle gagnant est ré-entraîné avec pondération de classes (`compute_class_weights`), gradient clipping (`max_norm=1.0`), `ReduceLROnPlateau` et `EarlyStopping`.

In [ ]:
from IPython.display import Image, display as ipy_display
from src.metrics import plot_learning_curves
from src.trainer import compute_class_weights, train_classifier

final_cfg = cfg["part1_models"]["final_training"]
train_texts = train_df[text_col].astype(str).tolist()
train_labels = train_df["label"].astype(int).tolist()
val_texts = val_df[text_col].astype(str).tolist()
val_labels = val_df["label"].astype(int).tolist()
test_texts = test_df[text_col].astype(str).tolist()
test_labels = test_df["label"].astype(int).tolist()

train_ds = ArabicClassificationDataset(train_texts, train_labels, vocab=vocab, max_len=max_seq_len)
val_ds = ArabicClassificationDataset(val_texts, val_labels, vocab=vocab, max_len=max_seq_len)
test_ds = ArabicClassificationDataset(test_texts, test_labels, vocab=vocab, max_len=max_seq_len)

class_weights = compute_class_weights(train_labels, num_classes=num_classes)
print("Poids de classes utilises :", [round(float(w), 3) for w in class_weights])

trained_models = {}
histories = {}
train_meta = {}

for arch in cfg["part1_models"]["architectures"]:
    print(f"\n=== Entrainement final : {arch.upper()} ===")
    seed_everything(cfg["project"]["seed"])
    hp = best_configs[arch]
    bs = int(hp["batch_size"])

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, collate_fn=classification_collate_fn)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, collate_fn=classification_collate_fn)

    model = build_classifier(
        arch=arch,
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=int(hp["embedding_dim"]),
        hidden_dim=int(hp["hidden_dim"]),
        num_layers=int(hp["num_layers"]),
        dropout=float(hp["dropout"]),
        pooling=str(hp["pooling"]),
        pad_idx=PAD_IDX,
    )
    ckpt_path = paths["checkpoints"] / f"{arch}_classifier_best.pt"
    res = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        epochs=int(final_cfg["epochs"]),
        lr=float(hp["lr"]),
        weight_decay=float(final_cfg["weight_decay"]),
        grad_clip=float(final_cfg["grad_clip"]),
        patience=int(final_cfg["patience"]),
        class_weights=class_weights,
        checkpoint_path=ckpt_path,
        verbose=True,
    )
    trained_models[arch] = model
    histories[arch] = res["history"]
    train_meta[arch] = {
        "best_epoch": res["best_epoch"],
        "best_val_f1": res["best_val_f1"],
        "train_time_sec": res["train_time_sec"],
        "n_params": model.count_parameters(),
    }

lc_fig = plot_learning_curves(histories, paths["figures"] / "part1_learning_curves.png")
ipy_display(Image(filename=str(lc_fig)))

## 5. Évaluation Classification (Métriques Standards & Analyse d'Erreurs)

In [ ]:
from src.metrics import compute_classification_metrics, plot_confusion_matrices
from src.trainer import predict_classifier

cls_results = {}
cls_summary_rows = []
predictions_cache = {}

test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=classification_collate_fn)

for arch, model in trained_models.items():
    pred_out = predict_classifier(model, test_loader, device=device)
    predictions_cache[arch] = pred_out
    metrics = compute_classification_metrics(
        y_true=pred_out["y_true"],
        y_pred=pred_out["y_pred"],
        y_prob=pred_out["y_prob"],
        class_names=class_names,
    )
    cls_results[arch] = metrics
    cls_summary_rows.append(
        {
            "arch": arch,
            "n_params": train_meta[arch]["n_params"],
            "best_epoch": train_meta[arch]["best_epoch"],
            "accuracy": metrics["accuracy"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
            "roc_auc_macro": metrics["roc_auc_macro"],
            "latency_ms_per_sample": pred_out["latency_ms_per_sample"],
            "train_time_sec": train_meta[arch]["train_time_sec"],
        }
    )

cls_df = pd.DataFrame(cls_summary_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(cls_df.round(4))

cm_fig = plot_confusion_matrices(cls_results, class_names=class_names, save_path=paths["figures"] / "part1_confusion_matrices.png")
ipy_display(Image(filename=str(cm_fig)))

In [ ]:
# Analyse qualitative des erreurs sur le meilleur modele
best_arch = str(cls_df.iloc[0]["arch"])
best_pred = predictions_cache[best_arch]
err_df = test_df.copy()
err_df["y_true"] = best_pred["y_true"]
err_df["y_pred"] = best_pred["y_pred"]
err_df["confidence"] = best_pred["y_prob"].max(axis=1)

misclassified = err_df[err_df["y_true"] != err_df["y_pred"]].sort_values("confidence", ascending=False)
print(f"Meilleur modele : {best_arch.upper()} | Erreurs sur test : {len(misclassified)} / {len(err_df)}")
display(misclassified[["title", "score", "y_true", "y_pred", "confidence", "text"]].head(5))

## 6. Benchmark Language Modeling & Score BLEU (`RNN`, `Bi-RNN`, `GRU`, `LSTM`)

> **Précision méthodologique importante :** Le score **BLEU** mesure le recouvrement n-gramme entre des *séquences textuelles générées* et des *séquences de référence* ; il ne s'applique donc pas à des étiquettes scalaires (`0/1/2`). Pour satisfaire rigoureusement la consigne (*"Evaluate the four languages models by using standards metrics and other metrics like bleu score"*), nous entraînons en parallèle les 4 mêmes architectures récurrentes comme **modèles de langage au niveau mot** sur le corpus arabe :
> - **`RNN`, `GRU`, `LSTM`** (causaux / unidirectionnels) : évalués en **Perplexity** + **BLEU-1..4 / SacreBLEU** sur la complétion autorégressive de préfixes issus du jeu de test.
> - **`Bi-RNN`** (bidirectionnel / non-causal) : ne pouvant générer de manière causale sans voir le futur, il est évalué via un objectif **Masked Cloze (15 % `<MASK>`)** rapportant la **Pseudo-Perplexité** et le **BLEU de reconstruction**.

In [ ]:
from src.metrics import plot_model_comparison
from src.trainer import train_and_evaluate_lm

lm_results = []
for arch in cfg["part1_models"]["architectures"]:
    print(f">>> Benchmark Language Model : {arch.upper()}...")
    res = train_and_evaluate_lm(
        arch=arch,
        train_texts=train_texts,
        val_texts=val_texts,
        test_texts=test_texts,
        vocab=vocab,
        cfg=cfg,
        device=device,
    )
    lm_results.append(res)

lm_df = pd.DataFrame(
    [
        {
            "arch": r["arch"],
            "evaluation_mode": r["evaluation_mode"],
            "test_loss": r["test_loss"],
            "perplexity": r["perplexity"],
            "bleu_1": r["bleu_1"],
            "bleu_2": r["bleu_2"],
            "bleu_4": r["bleu_4"],
            "sacrebleu": r["sacrebleu"],
            "n_eval_samples": r["n_eval_samples"],
        }
        for r in lm_results
    ]
)
display(lm_df.round(4))

comp_fig = plot_model_comparison(cls_df=cls_df, lm_df=lm_df, save_path=paths["figures"] / "part1_model_comparison.png")
ipy_display(Image(filename=str(comp_fig)))

# Sauvegarde du rapport JSON complet de la Partie 1
part1_report = {
    "best_hyperparams": best_configs,
    "classification_summary": cls_summary_rows,
    "classification_per_class": {
        arch: res["classification_report"] for arch, res in cls_results.items()
    },
    "language_modeling_bleu": lm_results,
}
(paths["reports"] / "metrics_part1.json").write_text(
    json.dumps(part1_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Rapport Partie 1 sauvegarde dans {paths['reports'] / 'metrics_part1.json'}")